In [ ]:
#!/usr/bin/env python3
"""
analyze_and_plot.py
- Reads processed outputs
- Computes metrics.json
- Generates 2-3 PNG charts under plots/
"""

# Install necessary libraries (no requirements.txt)
import sys
import subprocess
import importlib

def _ensure(import_name: str, pip_name: str) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--user", pip_name])

_ensure("pandas", "pandas")
_ensure("matplotlib", "matplotlib")

# Import libraries
import os
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def main() -> None:
    processed_dir = Path(os.getenv("TAXI_PROCESSED_DIR", "processed"))
    plots_dir = Path(os.getenv("TAXI_PLOTS_DIR", "plots"))
    metrics_dir = Path(os.getenv("TAXI_METRICS_DIR", "metrics"))
    plots_dir.mkdir(parents=True, exist_ok=True)
    metrics_dir.mkdir(parents=True, exist_ok=True)

    clean = pd.read_csv(processed_dir / "trips_clean.csv")
    agg = pd.read_csv(processed_dir / "aggregates_by_hour.csv")

    metrics = {
        "trips": int(len(clean)),
        "total_revenue": float(clean["total_amount"].sum()) if "total_amount" in clean.columns else None,
        "avg_total_amount": float(clean["total_amount"].mean()) if "total_amount" in clean.columns else None,
        "avg_distance_mi": float(clean["trip_distance"].mean()) if "trip_distance" in clean.columns else None,
        "avg_duration_min": float(clean["trip_duration_min"].mean()) if "trip_duration_min" in clean.columns else None,
    }

    if "payment_type" in clean.columns:
        counts = clean["payment_type"].value_counts(dropna=False).to_dict()
        metrics["payment_type_counts"] = {str(k): int(v) for k, v in counts.items()}

    (metrics_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    print(f"Wrote {metrics_dir / 'metrics.json'}", flush=True)

    # Plot 1: trip distance histogram
    if "trip_distance" in clean.columns:
        max_dist = float(os.getenv("TAXI_PLOT_MAX_DISTANCE", "15.0"))
        d = clean["trip_distance"].dropna()
        d = d[d <= max_dist]
        plt.figure()
        plt.hist(d, bins=30)
        plt.title("Trip distance distribution (miles)")
        plt.xlabel("Miles")
        plt.ylabel("Trips")
        out = plots_dir / "trip_distance_hist.png"
        plt.tight_layout()
        plt.savefig(out, dpi=150)
        plt.close()
        print(f"Wrote {out}", flush=True)

    # Plot 2: trip duration histogram
    if "trip_duration_min" in clean.columns:
        max_dur = float(os.getenv("TAXI_PLOT_MAX_DURATION", "60.0"))
        x = clean["trip_duration_min"].dropna()
        x = x[x <= max_dur]
        plt.figure()
        plt.hist(x, bins=30)
        plt.title("Trip duration distribution (minutes)")
        plt.xlabel("Minutes")
        plt.ylabel("Trips")
        out = plots_dir / "trip_duration_hist.png"
        plt.tight_layout()
        plt.savefig(out, dpi=150)
        plt.close()
        print(f"Wrote {out}", flush=True)

    # Plot 3: trips by pickup hour
    if "pickup_hour" in agg.columns and "trips" in agg.columns:
        plt.figure()
        plt.bar(agg["pickup_hour"], agg["trips"])
        plt.title("Trips by pickup hour")
        plt.xlabel("Hour")
        plt.ylabel("Trips")
        out = plots_dir / "trips_by_hour.png"
        plt.tight_layout()
        plt.savefig(out, dpi=150)
        plt.close()
        print(f"Wrote {out}", flush=True)

if __name__ == "__main__":
    main()
